# 🩺 Diabetic Retinopathy — Kaggle Top-1% Pipeline v21
## Regression + Multi-Model Ensemble + Pseudo Labels | Full Resume
### RTX 2050 (CUDA) + MPS + CPU | QWK Target ≥ 0.93
---
**Fix:** CUDA detection, PyTorch+CUDA install, kaggle.json upload, albumentations API


## 🔁 Step 0 — Global Resume System + Directories


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 0 — GLOBAL RESUME SYSTEM
# ═══════════════════════════════════════════════════════════════
import os, json, pickle, time, sys, platform, subprocess
from pathlib import Path

HOME = Path.home()
BASE_DIR = Path(os.environ.get("DR_BASE", str(HOME / "DR_data")))
DATA_DIR = BASE_DIR / "aptos2019"; FLAG_DIR = BASE_DIR / "flags"
CKPT_DIR = BASE_DIR / "checkpoints"; LOG_DIR = BASE_DIR / "logs"
CACHE_DIR = BASE_DIR / "cache"; ARTIFACT_DIR = BASE_DIR / "artifacts"
PLOT_DIR = BASE_DIR / "plots"; EXPORT_DIR = BASE_DIR / "export"
DEPLOY_DIR = BASE_DIR / "deploy"; IMG_DIR = DATA_DIR / "train_images"
CSV_PATH = DATA_DIR / "train.csv"

for d in [DATA_DIR, FLAG_DIR, CKPT_DIR, LOG_DIR, CACHE_DIR,
          ARTIFACT_DIR, PLOT_DIR, EXPORT_DIR, DEPLOY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def is_done(s): return (FLAG_DIR / f"{s}.done").exists()
def mark_done(s): (FLAG_DIR / f"{s}.done").touch()
def clear_done(s):
    f = FLAG_DIR / f"{s}.done"
    if f.exists(): f.unlink()
def save_json(d, p): Path(p).write_text(json.dumps(d, indent=2, default=str))
def load_json(p): return json.loads(Path(p).read_text())
def save_pickle(o, p):
    with open(p, "wb") as f: pickle.dump(o, f)
def load_pickle(p):
    with open(p, "rb") as f: return pickle.load(f)

_STATE = ARTIFACT_DIR / "state.json"
def st_load():
    if _STATE.exists():
        try: return json.loads(_STATE.read_text())
        except: return {}
    return {}
def st_save(k, v): s = st_load(); s[k] = v; _STATE.write_text(json.dumps(s, indent=2, default=str))
def st_get(k, d=None): return st_load().get(k, d)

def progress(cur, tot, pre="", w=50, ex=""):
    p = cur/max(tot,1)*100; f = int(w*cur//max(tot,1))
    print(f"\r  {pre} [{'█'*f}{'░'*(w-f)}] {p:5.1f}% ({cur:,}/{tot:,}) {ex}", end="", flush=True)
    if cur >= tot: print()

def step_start(n, name):
    print("=" * 68); print(f"  [STEP {n:02d}] {name}"); print("=" * 68)
    return time.time()

def step_skip(n, name, detail=""):
    print("=" * 68); print(f"  [STEP {n:02d}] {name}")
    print(f"  ✅ ALREADY COMPLETED → Skipping")
    if detail: print(f"  Detail: {detail}")
    print("=" * 68)

def step_end(n, t0, outs=None):
    e = time.time() - t0
    print(f"\n  ✅ Completed in {e:.1f}s")
    if outs:
        for k, v in outs.items(): print(f"  {k:8s}: {v}")
    print("=" * 68)

# Constants
NUM_CLASSES = 5; N_FOLDS = 5; SEED = 42
GRADE_MAP = {0:"No DR", 1:"Mild", 2:"Moderate", 3:"Severe", 4:"Proliferative"}
GRADE_COLORS = ["#2ecc71","#f1c40f","#e67e22","#e74c3c","#8e44ad"]
IMAGENET_MEAN = [0.485,0.456,0.406]; IMAGENET_STD = [0.229,0.224,0.225]
# REGRESSION thresholds (top-1% Kaggle)
REG_THRESHOLDS = [0.7, 1.5, 2.5, 3.5]

existing = sorted(FLAG_DIR.glob("*.done"))
print(f"  BASE: {BASE_DIR}")
print(f"  Completed steps: {len(existing)}")
for f in existing: print(f"    ✅ {f.stem}")
print("=" * 68)

## ⚙️ Step 1 — System Setup + CUDA Detection Fix
**Critical:** Detects GPU, checks CUDA toolkit, diagnoses why GPU might not be detected.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 — System Setup + CUDA/GPU DIAGNOSIS
# Fixes: PyTorch not detecting RTX 2050, CUDA version mismatch
# ═══════════════════════════════════════════════════════════════
import subprocess, platform, shutil

if is_done("system"):
    info = load_json(LOG_DIR / "system_info.json")
    step_skip(1, "SYSTEM SETUP", f"Device: {info.get('device','?')}")
else:
    t0 = step_start(1, "SYSTEM SETUP + CUDA DIAGNOSIS")

    info = {"os": f"{platform.system()} {platform.release()}", "python": sys.version.split()[0]}

    # RAM
    try:
        import psutil
        ram = psutil.virtual_memory()
        info["ram_gb"] = round(ram.total / 1e9, 1)
        print(f"  RAM       : {info['ram_gb']} GB")
    except: pass

    # Disk
    _, _, free = shutil.disk_usage(str(HOME))
    info["disk_free_gb"] = round(free / 1e9, 1)
    print(f"  Disk free : {info['disk_free_gb']} GB")

    # ── NVIDIA GPU Detection ─────────────────────────────────
    print("\n  ── GPU DETECTION ──")
    gpu_found = False
    try:
        r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=10)
        if r.returncode == 0:
            print(f"  nvidia-smi: ✅ FOUND")
            for line in r.stdout.split("\n"):
                if "RTX" in line or "GTX" in line or "NVIDIA" in line:
                    if "Driver" in line or "MiB" in line:
                        print(f"    {line.strip()}")
            gpu_found = True
            # Extract CUDA version from nvidia-smi
            for line in r.stdout.split("\n"):
                if "CUDA Version" in line:
                    import re
                    m = re.search(r"CUDA Version:\s*([\d.]+)", line)
                    if m: info["nvidia_cuda"] = m.group(1)
                    print(f"  NVIDIA CUDA: {info.get('nvidia_cuda','?')}")
        else:
            print(f"  nvidia-smi: ❌ NOT FOUND or FAILED")
            print(f"    → Install NVIDIA drivers from nvidia.com")
    except FileNotFoundError:
        print(f"  nvidia-smi: ❌ NOT FOUND")
        print(f"    → NVIDIA drivers not installed")
        print(f"    → Download from: https://www.nvidia.com/drivers")

    # ── PyTorch CUDA Check ────────────────────────────────────
    print("\n  ── PyTorch CUDA CHECK ──")
    try:
        import torch
        info["pytorch"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        info["cuda_built"] = torch.version.cuda or "None"

        print(f"  PyTorch     : {torch.__version__}")
        print(f"  Built with  : CUDA {torch.version.cuda}" if torch.version.cuda else "  Built with  : ❌ CPU-ONLY BUILD")
        print(f"  CUDA avail  : {torch.cuda.is_available()}")

        if torch.cuda.is_available():
            info["device"] = "cuda"
            info["gpu_name"] = torch.cuda.get_device_name(0)
            info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1)
            print(f"  GPU         : {info['gpu_name']}")
            print(f"  VRAM        : {info['vram_gb']} GB")
            print(f"  ✅ CUDA is working!")
        elif gpu_found and not torch.cuda.is_available():
            print(f"\n  ⚠️  GPU DETECTED but PyTorch can't use it!")
            print(f"  DIAGNOSIS:")
            if not torch.version.cuda:
                print(f"    → PyTorch was installed WITHOUT CUDA support")
                print(f"    → FIX: Run Step 2 to install correct PyTorch+CUDA")
            else:
                print(f"    → PyTorch built with CUDA {torch.version.cuda}")
                print(f"    → Driver CUDA: {info.get('nvidia_cuda','?')}")
                print(f"    → Version mismatch? Run Step 2 to fix")
            info["device"] = "cpu"
            info["cuda_issue"] = "pytorch_no_cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            info["device"] = "mps"
            print(f"  ✅ Apple MPS available")
        else:
            info["device"] = "cpu"
            print(f"  Running on CPU")

    except ImportError:
        print(f"  PyTorch: NOT INSTALLED (Step 2 will install)")
        info["device"] = "unknown"

    # Reproducibility
    import random, numpy as np
    random.seed(SEED); np.random.seed(SEED)
    if "torch" in dir():
        import torch
        torch.manual_seed(SEED)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(SEED)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    save_json(info, LOG_DIR / "system_info.json")
    mark_done("system")
    step_end(1, t0, {"Device": info.get("device","?")})

## 📦 Step 2 — Install Requirements + PyTorch CUDA Fix
**Fixes:** Installs PyTorch with correct CUDA version for RTX 2050.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 2 — Install + PyTorch CUDA Fix
# RTX 2050 needs: PyTorch with CUDA 11.8 or 12.1+
# ═══════════════════════════════════════════════════════════════
import importlib

if is_done("install"):
    step_skip(2, "INSTALL REQUIREMENTS")
else:
    t0 = step_start(2, "INSTALL REQUIREMENTS + CUDA FIX")

    # ── Fix 1: Install PyTorch with CUDA ─────────────────────
    print("  ── Checking PyTorch CUDA ──")
    need_pytorch_reinstall = False
    try:
        import torch
        if not torch.cuda.is_available():
            # Check if GPU exists but PyTorch lacks CUDA
            try:
                r = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=5)
                if r.returncode == 0:
                    need_pytorch_reinstall = True
                    print("  ⚠️ GPU found but PyTorch has no CUDA → reinstalling")
            except: pass
    except ImportError:
        need_pytorch_reinstall = True
        print("  PyTorch not installed → installing with CUDA")

    if need_pytorch_reinstall and platform.system() == "Windows":
        print("  📦 Installing PyTorch with CUDA 12.4 (Windows)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
            "torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu124"],
            capture_output=True, text=True)
        # Verify
        importlib.invalidate_caches()
        try:
            import importlib
            if "torch" in sys.modules: del sys.modules["torch"]
            import torch
            print(f"  PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()}")
            if torch.cuda.is_available():
                print(f"  ✅ GPU: {torch.cuda.get_device_name(0)}")
            else:
                print("  ⚠️ Still no CUDA. Try manually:")
                print("     pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121")
        except Exception as e:
            print(f"  ❌ Error: {e}")
    elif need_pytorch_reinstall and platform.system() == "Linux":
        print("  📦 Installing PyTorch with CUDA 12.4 (Linux)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
            "torch", "torchvision", "torchaudio",
            "--index-url", "https://download.pytorch.org/whl/cu124"],
            capture_output=True, text=True)
    elif need_pytorch_reinstall:
        print("  📦 Installing PyTorch (macOS — MPS)...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
            "torch", "torchvision", "torchaudio"], capture_output=True, text=True)
    else:
        print("  ✅ PyTorch+CUDA already working")

    # ── Fix 2: Install other packages ─────────────────────────
    _pkgs = {
        "timm":"timm>=1.0.0","albumentations":"albumentations>=1.4.0",
        "cv2":"opencv-python-headless","sklearn":"scikit-learn","scipy":"scipy",
        "pandas":"pandas","numpy":"numpy","tqdm":"tqdm","matplotlib":"matplotlib",
        "pytorch_grad_cam":"grad-cam","kaggle":"kaggle",
        "pyarrow":"pyarrow","PIL":"pillow<11.0","psutil":"psutil",
        "ipywidgets":"ipywidgets",
    }
    missing = []
    for mod, pkg in _pkgs.items():
        try: importlib.import_module(mod)
        except ImportError: missing.append(pkg)
    if missing:
        print(f"\n  Installing {len(missing)} missing packages...")
        for i, pkg in enumerate(missing):
            progress(i+1, len(missing), "Install", extra=pkg[:30])
            subprocess.run([sys.executable,"-m","pip","install","-q","--upgrade",pkg],
                           capture_output=True, text=True)
    else:
        print("  ✅ All packages installed")

    mark_done("install")
    step_end(2, t0)

## 🔑 Step 3 — Kaggle Authentication (Upload kaggle.json)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 3 — Kaggle Auth (Upload button + env vars + manual)
# ═══════════════════════════════════════════════════════════════
if is_done("kaggle_auth"):
    kj = Path.home() / ".kaggle" / "kaggle.json"
    u = "?"
    if kj.exists():
        try: u = json.loads(kj.read_text()).get("username","?")
        except: pass
    step_skip(3, "KAGGLE AUTH", f"User: {u}")
else:
    t0 = step_start(3, "KAGGLE AUTHENTICATION")
    KD = Path.home() / ".kaggle"; KJ = KD / "kaggle.json"

    if KJ.exists():
        d = json.loads(KJ.read_text())
        print(f"  ✅ Already configured — user: {d.get('username','?')}")
        mark_done("kaggle_auth")
    else:
        ku = os.environ.get("KAGGLE_USERNAME",""); kk = os.environ.get("KAGGLE_KEY","")
        if ku and kk:
            KD.mkdir(parents=True, exist_ok=True)
            KJ.write_text(json.dumps({"username":ku,"key":kk}))
            if platform.system() != "Windows": KJ.chmod(0o600)
            print(f"  ✅ From env vars — user: {ku}")
            mark_done("kaggle_auth")
        else:
            try:
                import ipywidgets as widgets
                from IPython.display import display, HTML
                display(HTML("<h4>📁 Upload kaggle.json:</h4>"))
                _up = widgets.FileUpload(accept=".json", multiple=False)
                _lbl = widgets.Label("⏳ Waiting for upload...")
                def _on(change):
                    if _up.value:
                        uv = list(_up.value.values())[0] if isinstance(_up.value, dict) else _up.value[0]
                        ct = uv["content"] if isinstance(uv, dict) else uv.content
                        KD.mkdir(parents=True, exist_ok=True)
                        KJ.write_bytes(ct if isinstance(ct, bytes) else ct.tobytes())
                        if platform.system() != "Windows": KJ.chmod(0o600)
                        d2 = json.loads(KJ.read_text())
                        _lbl.value = f"✅ User: {d2.get('username','?')}"
                        mark_done("kaggle_auth")
                _up.observe(_on, names="value")
                display(_up, _lbl)
                display(HTML(f"<p>Or place at: <code>{KJ}</code></p>"))
            except ImportError:
                print(f"  Place kaggle.json at: {KJ}")

    if KJ.exists(): os.environ["KAGGLE_CONFIG_DIR"] = str(KD)
    step_end(3, t0)

## 📥 Step 4 — Dataset Download & Extraction


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 4 — Download + Extract (RESUMABLE per sub-step)
# ═══════════════════════════════════════════════════════════════
import zipfile

if is_done("extract"):
    n = len(list(IMG_DIR.glob("*.png"))) if IMG_DIR.exists() else 0
    step_skip(4, "DATASET DOWNLOAD", f"{n:,} images")
else:
    t0 = step_start(4, "DATASET DOWNLOAD & EXTRACTION")
    KJ = Path.home() / ".kaggle" / "kaggle.json"
    if not KJ.exists():
        raise FileNotFoundError(f"Run Step 3 first. Need: {KJ}")

    ZIP = DATA_DIR / "aptos2019-blindness-detection.zip"
    if not is_done("download"):
        if not ZIP.exists():
            print("  📥 Downloading APTOS 2019...")
            r = subprocess.run([sys.executable,"-m","kaggle","competitions","download",
                "-c","aptos2019-blindness-detection","-p",str(DATA_DIR)],
                capture_output=True, text=True)
            if r.returncode != 0:
                print(f"  ❌ {r.stderr[-500:]}")
                raise RuntimeError("Download failed. Accept competition rules at kaggle.com")
        else:
            print(f"  ✅ ZIP exists")
        mark_done("download")

    # Find ZIP
    if not ZIP.exists():
        zips = list(DATA_DIR.glob("*.zip"))
        if zips: ZIP = zips[0]
        else: raise FileNotFoundError(f"No ZIP found in {DATA_DIR}")

    print(f"  📦 Extracting {ZIP.name}...")
    with zipfile.ZipFile(ZIP,"r") as zf:
        members = zf.namelist(); total = len(members)
        for i, m in enumerate(members):
            zf.extract(m, DATA_DIR)
            if (i+1)%max(1,total//30)==0 or i==total-1:
                progress(i+1, total, "Extract")

    mark_done("extract")
    n = len(list(IMG_DIR.glob("*.png"))) if IMG_DIR.exists() else 0
    step_end(4, t0, {"Images": f"{n:,}"})

## 📊 Step 5 — Load Dataset + ALL Imports (Master Reload)
**Run after any kernel restart to restore full state.**


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 5 — MASTER RELOAD (always runs — fast)
# Fix: reload base CSV first → overlay clean → overlay kfold
# ═══════════════════════════════════════════════════════════════
import os, sys, io, json, gc, time, random, shutil, warnings, pickle, platform, re
from pathlib import Path
from copy import deepcopy
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm
from scipy.optimize import minimize

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (confusion_matrix, classification_report,
    cohen_kappa_score, ConfusionMatrixDisplay,
    precision_score, recall_score, f1_score, roc_auc_score)

warnings.filterwarnings("ignore")
t0 = step_start(5, "MASTER RELOAD")

# ── Reproducibility ───────────────────────────────────────────
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed); torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything()

# ── Device (FIXED) ────────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    USE_AMP = True
    PIN_MEM = True
    print(f"  🔥 CUDA: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB)")
elif hasattr(torch.backends,"mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    USE_AMP = False  # MPS: no AMP
    PIN_MEM = False  # MPS: no pin_memory
    print(f"  🍎 MPS")
else:
    DEVICE = "cpu"
    USE_AMP = False
    PIN_MEM = False
    print(f"  💻 CPU")
print(f"  PyTorch {torch.__version__} | AMP: {USE_AMP} | Device: {DEVICE}")

# ── Safe load ─────────────────────────────────────────────────
def safe_load(path, ml="cpu"):
    try: return torch.load(path, map_location=ml, weights_only=False)
    except TypeError: return torch.load(path, map_location=ml)

# ── num_workers (macOS fix) ───────────────────────────────────
IS_MAC = platform.system() == "Darwin"
IS_WIN = platform.system() == "Windows"
NW = 0 if (IS_MAC or IS_WIN) else min(4, os.cpu_count() or 1)

# ── Load CSV (FIX: base → clean → kfold overlay) ─────────────
if not CSV_PATH.exists():
    raise FileNotFoundError(f"train.csv not found. Run Step 4.")
df = pd.read_csv(CSV_PATH)
df["image_path"] = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
df["binary"] = (df["diagnosis"] >= 1).astype(int)
print(f"  Base CSV: {len(df):,} rows")

# Overlay clean
_cp = ARTIFACT_DIR / "df_clean.parquet"
if is_done("cleaning") and _cp.exists():
    df = pd.read_parquet(_cp)
    df["image_path"] = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"] = (df["diagnosis"] >= 1).astype(int)
    print(f"  Clean:    {len(df):,} rows")

# Overlay kfold
_sp = ARTIFACT_DIR / "kfold_splits.parquet"
if is_done("kfold") and _sp.exists():
    df = pd.read_parquet(_sp)
    df["image_path"] = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"] = (df["diagnosis"] >= 1).astype(int)
    print(f"  K-Fold:   {len(df):,} rows, {N_FOLDS} folds")

# Cache check
USE_CACHE = is_done("preprocess") and CACHE_DIR.exists()
CACHE_SIZE = 384

print(f"\n  Dataset: {len(df):,} | Classes: {NUM_CLASSES} | Cache: {USE_CACHE}")
for g in range(5):
    n = (df["diagnosis"]==g).sum()
    print(f"    G{g} ({GRADE_MAP[g]:14s}): {n:5d} {'█'*(n//60)}")

mark_done("data_load")
step_end(5, t0)

## 🧹 Step 6 — Data Cleaning (CORRECTED: minimal removal only)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 6 — Data Cleaning
# FIX: ONLY remove unreadable, tiny (<50px), completely black
# DO NOT remove blur/color — those are valid fundus images!
# ═══════════════════════════════════════════════════════════════
if is_done("cleaning"):
    df = pd.read_parquet(ARTIFACT_DIR / "df_clean.parquet")
    df["image_path"] = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"] = (df["diagnosis"] >= 1).astype(int)
    step_skip(6, "DATA CLEANING", f"{len(df):,} rows")
else:
    t0 = step_start(6, "DATA CLEANING (minimal)")
    total = len(df); bad = []
    for i, (_, row) in enumerate(df.iterrows()):
        bgr = cv2.imread(str(row["image_path"]))
        reason = None
        if bgr is None: reason = "unreadable"
        elif bgr.shape[0] < 50 or bgr.shape[1] < 50: reason = "tiny"
        elif float(cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY).mean()) < 3: reason = "black"
        if reason: bad.append((row["id_code"], reason))
        if (i+1)%max(1,total//30)==0 or i==total-1:
            progress(i+1, total, "Checking")
    print(f"\n  Removed: {len(bad)}")
    for r in set(x[1] for x in bad):
        print(f"    {r}: {sum(1 for x in bad if x[1]==r)}")
    bad_ids = set(x[0] for x in bad)
    df = df[~df["id_code"].isin(bad_ids)].reset_index(drop=True)
    df.to_parquet(ARTIFACT_DIR / "df_clean.parquet", index=False)
    mark_done("cleaning")
    step_end(6, t0, {"Clean": f"{len(df):,}"})

## 📈 Step 7 — EDA


In [ ]:
# STEP 7 — EDA (RESUMABLE)
if is_done("eda"):
    step_skip(7, "EDA")
else:
    t0 = step_start(7, "EDA")
    %matplotlib inline
    fig,axes=plt.subplots(1,2,figsize=(14,5))
    counts=df["diagnosis"].value_counts().sort_index()
    axes[0].bar([GRADE_MAP[i] for i in range(5)],counts.values,color=GRADE_COLORS,edgecolor="k")
    axes[0].set_title("Class Distribution",fontweight="bold")
    for i,v in enumerate(counts.values): axes[0].text(i,v+20,str(v),ha="center")
    axes[1].pie(counts.values,labels=[GRADE_MAP[i] for i in range(5)],colors=GRADE_COLORS,autopct="%1.1f%%")
    plt.suptitle(f"APTOS 2019 — {len(df):,} images",fontweight="bold")
    plt.tight_layout(); plt.savefig(PLOT_DIR/"eda.png",dpi=150); plt.show()
    mark_done("eda")
    step_end(7, t0)

## ⚖️ Step 8 — Label Analysis


In [ ]:
# STEP 8 — Label Analysis (RESUMABLE)
if is_done("label_analysis"):
    step_skip(8, "LABEL ANALYSIS")
else:
    t0 = step_start(8, "LABEL ANALYSIS")
    vc=df["diagnosis"].value_counts().sort_index()
    c=np.bincount(df["diagnosis"].values,minlength=5).astype(float)
    w=len(df)/(5*np.maximum(c,1)); w=w/w.sum()*5
    for g,cnt in vc.items():
        print(f"  G{g}: {cnt:5d} ({cnt/len(df)*100:5.1f}%) weight={w[g]:.3f}")
    save_json({"weights":{str(i):round(float(w[i]),4) for i in range(5)}},LOG_DIR/"weights.json")
    mark_done("label_analysis")
    step_end(8, t0)

## 🧠 Step 9 — Preprocessing (Top-1%: resize only, NO CLAHE)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 9 — Preprocessing (CORRECTED per top-1% strategy)
# STRICT: Resize only + normalize. NO CLAHE, NO heavy enhancement
# ═══════════════════════════════════════════════════════════════
t0 = step_start(9, "PREPROCESSING PIPELINE")

IMG_SIZE = int(os.environ.get("IMG_SIZE", 512))

def preprocess_fundus(path_or_arr, size=None):
    target = size or IMG_SIZE
    if isinstance(path_or_arr, np.ndarray):
        rgb = path_or_arr.copy()
    else:
        bgr = cv2.imread(str(path_or_arr))
        if bgr is None: return np.zeros((target,target,3), np.uint8)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    # Simple crop black borders
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, th = cv2.threshold(gray, 7, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(th)
    if coords is not None:
        x,y,w,h = cv2.boundingRect(coords); rgb = rgb[y:y+h, x:x+w]
    # Resize to square (simple)
    rgb = cv2.resize(rgb, (target, target), interpolation=cv2.INTER_AREA)
    return rgb

print(f"  Strategy: Crop borders → Resize {IMG_SIZE}px (NO CLAHE, NO enhancement)")
if len(df)>0:
    _t=time.time()
    for _ in range(3): preprocess_fundus(df["image_path"].iloc[0])
    print(f"  Latency : {(time.time()-_t)/3*1000:.0f}ms")
step_end(9, t0)

## 💾 Step 10 — Preprocessing Cache


In [ ]:
# STEP 10 — Cache (RESUMABLE: skips cached)
if is_done("preprocess"):
    n=len(list(CACHE_DIR.glob("*.npy"))); USE_CACHE=True
    step_skip(10, "CACHE", f"{n:,} files")
else:
    t0 = step_start(10, "PREPROCESSING CACHE")
    existing=set(p.stem for p in CACHE_DIR.glob("*.npy"))
    todo=df[~df["id_code"].isin(existing)]
    print(f"  Cached: {len(existing):,} | Todo: {len(todo):,}")
    ok=0
    for i,(_, row) in enumerate(todo.iterrows()):
        try:
            img=preprocess_fundus(row["image_path"],size=CACHE_SIZE)
            np.save(str(CACHE_DIR/f'{row["id_code"]}.npy'),img); ok+=1
        except: pass
        if (i+1)%max(1,len(todo)//30)==0 or i==len(todo)-1:
            progress(i+1, len(todo), "Caching")
    USE_CACHE=True; mark_done("preprocess")
    step_end(10, t0, {"Total":f"{ok+len(existing):,}"})

## ✂️ Step 11 — Train/Test Split


In [ ]:
# STEP 11
if is_done("split"):
    step_skip(11, "TRAIN/TEST SPLIT")
else:
    t0=step_start(11,"TRAIN/TEST SPLIT")
    print("  Fold 0 = hold-out test (never trained)"); mark_done("split"); step_end(11,t0)

## 🔁 Step 12 — Stratified K-Fold


In [ ]:
# STEP 12 (RESUMABLE)
if is_done("kfold"):
    df=pd.read_parquet(ARTIFACT_DIR/"kfold_splits.parquet")
    df["image_path"]=df["id_code"].apply(lambda x:str(IMG_DIR/f"{x}.png"))
    df["grade_label"]=df["diagnosis"].map(GRADE_MAP); df["binary"]=(df["diagnosis"]>=1).astype(int)
    step_skip(12, "K-FOLD", f"{len(df):,} rows")
else:
    t0=step_start(12,"K-FOLD")
    skf=StratifiedKFold(n_splits=N_FOLDS,shuffle=True,random_state=SEED)
    df["fold"]=-1
    for fi,(_,vi) in enumerate(skf.split(df,df["diagnosis"])): df.loc[vi,"fold"]=fi
    df.to_parquet(ARTIFACT_DIR/"kfold_splits.parquet",index=False)
    for f in range(N_FOLDS):
        n=(df["fold"]==f).sum()
        print(f"  Fold {f}: {n}")
    mark_done("kfold"); step_end(12,t0)

## 🔧 Step 13 — Augmentation + Dataset (albumentations API FIXED)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 13 — Augmentation + Dataset
# FIX: RandomResizedCrop(size=(sz,sz)), CoarseDropout(num_holes_range=)
# FIX: macOS num_workers=0, pin_memory=False
# APPROACH: Regression (output=1 value, not 5 classes)
# ═══════════════════════════════════════════════════════════════
t0 = step_start(13, "AUGMENTATION + DATASET")

def build_train_tf(sz=IMG_SIZE):
    return A.Compose([
        A.RandomResizedCrop(size=(sz,sz), scale=(0.8,1.0), ratio=(0.9,1.1), p=1.0),
        A.HorizontalFlip(p=0.5), A.VerticalFlip(p=0.3),
        A.Rotate(limit=180, p=0.7),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=45, p=0.5),
        A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
        A.HueSaturationValue(10, 20, 10, p=0.3),
        A.OneOf([A.GaussianBlur(blur_limit=(3,5)), A.Sharpen()], p=0.3),
        A.CoarseDropout(num_holes_range=(1,8),
                        hole_height_range=(sz//20, sz//10),
                        hole_width_range=(sz//20, sz//10), p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD), ToTensorV2(),
    ])

def build_val_tf(sz=IMG_SIZE):
    return A.Compose([A.Resize(sz,sz),A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD),ToTensorV2()])

def build_tta_tf(sz=IMG_SIZE):
    n=A.Normalize(mean=IMAGENET_MEAN,std=IMAGENET_STD); r=A.Resize(sz,sz)
    return [build_val_tf(sz),
        A.Compose([A.HorizontalFlip(p=1),r,n,ToTensorV2()]),
        A.Compose([A.Rotate(limit=(10,10),p=1),r,n,ToTensorV2()]),
        A.Compose([A.Rotate(limit=(-10,-10),p=1),r,n,ToTensorV2()]),
        A.Compose([A.VerticalFlip(p=1),r,n,ToTensorV2()]),]

class DRDataset(Dataset):
    """Regression dataset: returns float label (0.0–4.0)."""
    def __init__(self,df,transform=None,img_size=None,use_cache=True):
        self.df=df.reset_index(drop=True); self.tf=transform
        self.sz=img_size or IMG_SIZE; self.cache=use_cache and USE_CACHE
    def __len__(self): return len(self.df)
    def __getitem__(self,idx):
        row=self.df.iloc[idx]; label=float(row["diagnosis"])  # REGRESSION
        if self.cache:
            cp=CACHE_DIR/f'{row["id_code"]}.npy'
            if cp.exists():
                img=np.load(str(cp))
                if self.sz!=CACHE_SIZE: img=cv2.resize(img,(self.sz,self.sz))
            else: img=preprocess_fundus(row["image_path"],self.sz)
        else: img=preprocess_fundus(row["image_path"],self.sz)
        if self.tf: img=self.tf(image=img)["image"]
        else: img=torch.from_numpy(img.transpose(2,0,1)).float()/255.0
        return img, torch.tensor(label, dtype=torch.float32)

def build_sampler(df_s):
    labs=df_s["diagnosis"].values; c=np.bincount(labs,minlength=5).astype(float)
    w=1.0/np.maximum(c,1); sw=w[labs]
    return WeightedRandomSampler(torch.DoubleTensor(sw),len(sw),replacement=True)

def make_loader(ds,bs=16,shuffle=True,sampler=None,drop_last=False):
    if sampler: shuffle=False
    return DataLoader(ds,batch_size=bs,shuffle=shuffle,sampler=sampler,
        num_workers=NW,pin_memory=PIN_MEM,persistent_workers=(NW>0),drop_last=drop_last)

BS_MAP = {224:16, 384:8, 512:2}  # RTX 2050 (4GB)
tta_tfs = build_tta_tf(IMG_SIZE)

def qwk(yt,yp): return cohen_kappa_score(yt,yp,weights="quadratic")
def acc_fn(yt,yp): return (np.array(yt)==np.array(yp)).mean()
def reg_to_class(preds, thresholds=REG_THRESHOLDS):
    """Convert regression output to class via thresholds."""
    p = np.clip(np.array(preds), 0, 4)
    return np.digitize(p, thresholds).astype(int)

mark_done("dataset")
print(f"  Approach   : REGRESSION (output=1, SmoothL1Loss)")
print(f"  Thresholds : {REG_THRESHOLDS}")
print(f"  API fixes  : RandomResizedCrop(size=), CoarseDropout(num_holes_range=)")
print(f"  Workers    : {NW} | pin_memory: {PIN_MEM}")
print(f"  Batch sizes: {BS_MAP}")
step_end(13, t0)

## 🚚 Step 14 — DataLoader Check


In [ ]:
# STEP 14
if is_done("dataloader"):
    step_skip(14, "DATALOADER")
else:
    t0=step_start(14,"DATALOADER CHECK")
    _d=DRDataset(df[df["fold"]!=0].head(64),build_val_tf(224),224)
    _l=make_loader(_d,bs=8)
    im,lb=next(iter(_l))
    print(f"  Batch: {im.shape} | Labels: {lb[:4].tolist()} (regression)")
    assert im.device.type=="cpu" and lb.dtype==torch.float32
    del _d,_l; gc.collect()
    mark_done("dataloader"); step_end(14,t0)

## 🧠 Step 15 — Model (Regression + GeM + Multi-backbone)


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 15 — Model Architecture (REGRESSION: output=1)
# ═══════════════════════════════════════════════════════════════
t0=step_start(15,"MODEL ARCHITECTURE")
BACKBONE = os.environ.get("BACKBONE", "tf_efficientnetv2_b1")

class GeM(nn.Module):
    def __init__(self,p=3,eps=1e-6):
        super().__init__(); self.p=nn.Parameter(torch.ones(1)*p); self.eps=eps
    def forward(self,x):
        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),(x.size(-2),x.size(-1))).pow(1./self.p)

class DRModelReg(nn.Module):
    """Regression model: backbone → GeM → Linear(256) → BN → ReLU → Drop → Linear(1)"""
    def __init__(self,backbone=BACKBONE,drop=0.5,pretrained=True):
        super().__init__()
        self.backbone=timm.create_model(backbone,pretrained=pretrained,num_classes=0,global_pool="")
        fd=self.backbone.num_features; self.pool=GeM()
        self.head=nn.Sequential(nn.Flatten(),nn.Linear(fd,256),nn.BatchNorm1d(256),
            nn.ReLU(True),nn.Dropout(drop),nn.Linear(256,1))
    def forward(self,x): return self.head(self.pool(self.backbone(x))).squeeze(-1)
    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)
    def unfreeze_top(self, n=4):
        for p in self.backbone.parameters(): p.requires_grad_(False)
        for b in list(self.backbone.blocks)[-n:]:
            for p in b.parameters(): p.requires_grad_(True)
    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad_(True)

def build_model(backbone=BACKBONE,pretrained=True):
    return DRModelReg(backbone,0.5,pretrained).to(DEVICE)

class EMA:
    def __init__(self,model,decay=0.9999):
        self.decay=decay
        self.shadow={n:p.clone().detach() for n,p in model.named_parameters() if p.requires_grad}
    def update(self,model):
        for n,p in model.named_parameters():
            if p.requires_grad and n in self.shadow:
                self.shadow[n].mul_(self.decay).add_(p.data,alpha=1-self.decay)
    def apply(self,model):
        self.backup={n:p.clone() for n,p in model.named_parameters() if n in self.shadow}
        for n,p in model.named_parameters():
            if n in self.shadow: p.data.copy_(self.shadow[n])
    def restore(self,model):
        for n,p in model.named_parameters():
            if n in self.backup: p.data.copy_(self.backup[n])

_m=build_model(pretrained=False)
print(f"  Backbone: {BACKBONE}")
print(f"  Output  : 1 (regression)")
print(f"  Loss    : SmoothL1Loss")
print(f"  Params  : {sum(p.numel() for p in _m.parameters())/1e6:.2f}M")
print(f"  EMA     : decay=0.9999")
del _m; gc.collect()
if DEVICE=="cuda": torch.cuda.empty_cache()
mark_done("model"); step_end(15,t0)

## ⚙️ Step 16 — Training Config


In [ ]:
# STEP 16 — Config
t0=step_start(16,"TRAINING CONFIG")
print("  Loss      : SmoothL1Loss (regression)")
print("  Optimizer : AdamW(lr=1e-4, wd=1e-4)")
print("  Scheduler : Warmup(5ep) + CosineAnnealing")
print("  Grad clip : 1.0")
print("  Grad accum: phase-dependent")
mark_done("training_config"); step_end(16,t0)

## 💾 Step 17 — Checkpoint System


In [ ]:
# STEP 17
t0=step_start(17,"CHECKPOINT SYSTEM")
print("  Saves: model, EMA, optimizer, scheduler, scaler, epoch, fold, phase, best_qwk, history")
print("  Resume: fold → phase → epoch")
mark_done("checkpoint"); step_end(17,t0)

## 🔁 Step 18 — Training State


In [ ]:
# STEP 18
t0=step_start(18,"TRAINING STATE")
st_save("version","v21"); mark_done("training_state"); step_end(18,t0)

## 🔬 Step 19 — Training (3-Phase × 5-Fold, Regression, EMA)
**Steps 20** (Validation), **21** (Early Stopping), **22** (OOF) integrated.
- Phase 1: 224px, 30ep, frozen backbone, bs=16
- Phase 2: 384px, 80ep, partial unfreeze, bs=8
- Phase 3: 512px, 50ep, full unfreeze, bs=2, grad_accum=4


In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEPS 19-22 — TRAINING (Regression, EMA, AMP, Full Resume)
# FIX: early_stopping patience=10, min_delta=0.0005
# FIX: LR lambda captures variables properly
# FIX: CUDA AMP only, MPS no AMP
# ═══════════════════════════════════════════════════════════════
WD=1e-4
PHASES=[
    {"size":224,"epochs":30,"name":"P1-Freeze","unfreeze":0,"lr":1e-4,"accum":1},
    {"size":384,"epochs":80,"name":"P2-Partial","unfreeze":4,"lr":5e-5,"accum":2},
    {"size":512,"epochs":50,"name":"P3-Full","unfreeze":99,"lr":1e-5,"accum":4},
]
ES_PAT=10; ES_DELTA=0.0005

if is_done("training"):
    oof_preds=np.load(str(ARTIFACT_DIR/"oof_preds.npy"))
    oof_labels=np.load(str(ARTIFACT_DIR/"oof_labels.npy"))
    fold_qwks=load_json(ARTIFACT_DIR/"fold_qwks.json")
    step_skip(19,"TRAINING",f"Mean QWK={np.mean(fold_qwks):.4f}")
else:
    t0=step_start(19,"5-FOLD × 3-PHASE REGRESSION TRAINING")
    oof_preds=np.zeros(len(df),dtype=np.float32)
    oof_labels=df["diagnosis"].values.copy()
    fold_qwks=[]
    criterion=nn.SmoothL1Loss()

    for fold in range(N_FOLDS):
        fc=CKPT_DIR/f"fold{fold}_best.pt"; fo=ARTIFACT_DIR/f"fold{fold}_oof.npy"
        ff=f"fold{fold}"
        if is_done(ff) and fc.exists():
            if fo.exists(): oof_preds[df[df["fold"]==fold].index]=np.load(str(fo))
            prev=safe_load(fc,"cpu"); fold_qwks.append(prev.get("val_qwk",0.0))
            print(f"  ✅ [RESUME] Fold {fold} QWK={fold_qwks[-1]:.4f}"); continue

        print(f"\n  {'═'*20} FOLD {fold} {'═'*20}")
        seed_everything(SEED+fold)
        df_tr=df[df["fold"]!=fold].reset_index(drop=True)
        df_va=df[df["fold"]==fold].reset_index(drop=True)
        vi=df[df["fold"]==fold].index
        model=build_model(pretrained=True); ema=EMA(model)
        best_qwk=-1.0; best_state=None

        for pi,phase in enumerate(PHASES):
            sz=phase["size"]; nep=phase["epochs"]; pn=phase["name"]
            unf=phase["unfreeze"]; plr=phase["lr"]; accum=phase["accum"]
            bs=BS_MAP[sz]; pc=CKPT_DIR/f"fold{fold}_p{pi}.pt"

            if unf==0: model.freeze_backbone()
            elif unf>=99: model.unfreeze_all()
            else: model.unfreeze_top(unf)
            tp=sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"  [{pn}] {sz}px ×{nep}ep bs={bs} accum={accum} | {tp/1e6:.2f}M")

            tr_ds=DRDataset(df_tr,build_train_tf(sz),sz)
            va_ds=DRDataset(df_va,build_val_tf(sz),sz)
            smp=build_sampler(df_tr)
            tr_ld=make_loader(tr_ds,bs,sampler=smp,drop_last=True)
            va_ld=make_loader(va_ds,bs,shuffle=False)

            opt=torch.optim.AdamW(filter(lambda p:p.requires_grad,model.parameters()),lr=plr,weight_decay=WD)
            # FIX: capture variables in lambda closure
            _warmup=5; _nep=nep
            def lr_fn(ep,warmup=_warmup,total=_nep):
                if ep<warmup: return (ep+1)/warmup
                return 0.5*(1+np.cos(np.pi*(ep-warmup)/(total-warmup)))
            sched=torch.optim.lr_scheduler.LambdaLR(opt,lr_fn)
            scaler=torch.amp.GradScaler("cuda") if USE_AMP else None

            ep_start=0
            if pc.exists():
                ps=safe_load(pc,DEVICE); model.load_state_dict(ps["model_state"])
                opt.load_state_dict(ps["optimizer_state"]); sched.load_state_dict(ps["scheduler_state"])
                ep_start=ps["epoch"]; best_qwk=ps.get("best_qwk",best_qwk)
                if ps.get("best_state"): best_state=ps["best_state"]
                if ps.get("ema"): ema.shadow=ps["ema"]
                print(f"    ↻ Resume ep {ep_start+1}/{nep}")

            model.to(DEVICE); es=0
            for ep in range(ep_start,nep):
                model.train(); el=0.0; opt.zero_grad()
                for step,(imgs,labs) in enumerate(tr_ld):
                    imgs,labs=imgs.to(DEVICE),labs.to(DEVICE)
                    if USE_AMP:
                        with torch.amp.autocast("cuda"):
                            out=model(imgs); loss=criterion(out,labs)/accum
                        scaler.scale(loss).backward()
                        if (step+1)%accum==0:
                            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                            scaler.step(opt); scaler.update(); opt.zero_grad()
                    else:
                        out=model(imgs); loss=criterion(out,labs)/accum
                        loss.backward()
                        if (step+1)%accum==0:
                            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                            opt.step(); opt.zero_grad()
                    el+=loss.item()*accum; ema.update(model)
                sched.step(); tl=el/max(len(tr_ld),1)

                # Validate with EMA
                ema.apply(model); model.eval()
                vp=[]; vl=[]
                with torch.no_grad():
                    for imgs,labs in va_ld:
                        imgs=imgs.to(DEVICE)
                        out=model(imgs).cpu().numpy()
                        vp.extend(out.tolist()); vl.extend(labs.numpy().tolist())
                ema.restore(model)

                vp_cls=reg_to_class(vp); vl_cls=np.array(vl,dtype=int)
                vk=qwk(vl_cls,vp_cls)

                flag=""
                if vk>best_qwk+ES_DELTA:
                    best_qwk=vk; es=0; flag=" ✅"
                    ema.apply(model); best_state=deepcopy(model.state_dict()); ema.restore(model)
                else: es+=1
                print(f"    Ep{ep+1:02d} TrL={tl:.4f} QWK={vk:.4f}{flag} ES={es}/{ES_PAT}")

                torch.save({"epoch":ep+1,"model_state":model.state_dict(),"best_state":best_state,
                    "optimizer_state":opt.state_dict(),"scheduler_state":sched.state_dict(),
                    "best_qwk":best_qwk,"ema":ema.shadow,
                    **({"scaler":scaler.state_dict()} if scaler else {})},pc)

                if es>=ES_PAT: print(f"    ⏹ Early stop"); break

            del tr_ds,va_ds,tr_ld,va_ld,opt,sched; gc.collect()
            if DEVICE=="cuda": torch.cuda.empty_cache()

        # OOF with EMA + TTA
        if best_state: model.load_state_dict(best_state)
        model.eval()
        oof_fold=np.zeros(len(df_va),dtype=np.float32)
        with torch.no_grad():
            for ttf in build_tta_tf(384):
                ds2=DRDataset(df_va,ttf,384); ld2=make_loader(ds2,BS_MAP[384],shuffle=False)
                bp=[]
                for im,_ in ld2: bp.extend(model(im.to(DEVICE)).cpu().numpy().tolist())
                oof_fold+=np.array(bp)
        oof_fold/=len(build_tta_tf(384))
        oof_preds[vi]=oof_fold; np.save(str(fo),oof_fold)
        fold_qwks.append(best_qwk)
        torch.save({"model_state":best_state,"val_qwk":best_qwk,"backbone":BACKBONE,
            "img_size":384,"fold":fold,"ema":ema.shadow},fc)
        mark_done(ff)
        print(f"  ✅ Fold {fold} QWK={best_qwk:.4f}")
        del model; gc.collect()
        if DEVICE=="cuda": torch.cuda.empty_cache()

    np.save(str(ARTIFACT_DIR/"oof_preds.npy"),oof_preds)
    np.save(str(ARTIFACT_DIR/"oof_labels.npy"),oof_labels)
    save_json(fold_qwks,ARTIFACT_DIR/"fold_qwks.json")
    oof_cls=reg_to_class(oof_preds); oq=qwk(oof_labels,oof_cls)
    st_save("oof_qwk",float(oq))
    mark_done("training")
    print("\n"+"="*68)
    for i,q in enumerate(fold_qwks): print(f"  Fold {i}: QWK={q:.4f}")
    print(f"  Mean: {np.mean(fold_qwks):.4f} ± {np.std(fold_qwks):.4f}")
    print(f"  OOF:  QWK={oq:.4f}")
    step_end(19,t0)

## 📊 Step 20 — Validation Summary


In [ ]:
# STEP 20
if is_done("training"):
    fq=load_json(ARTIFACT_DIR/"fold_qwks.json")
    for i,q in enumerate(fq): print(f"  Fold {i}: QWK={q:.4f}")
    print(f"  Mean: {np.mean(fq):.4f}"); mark_done("validation")
else: print("  Run Step 19 first")

## ⏹ Step 21 — Early Stopping


In [ ]:
# STEP 21
print(f"  Patience: {ES_PAT} | Delta: {ES_DELTA} | Integrated in Step 19")
mark_done("early_stopping")

## 📦 Step 22 — OOF Predictions


In [ ]:
# STEP 22
if is_done("training"):
    op=np.load(str(ARTIFACT_DIR/"oof_preds.npy")); ol=np.load(str(ARTIFACT_DIR/"oof_labels.npy"))
    oc=reg_to_class(op)
    print(f"  OOF QWK: {qwk(ol,oc):.4f} | Acc: {acc_fn(ol,oc)*100:.2f}%")
    mark_done("oof")
else: print("  Run Step 19")

## 🔁 Step 23 — TTA


In [ ]:
# STEP 23
print(f"  5-view TTA: original + HFlip + Rot±10 + VFlip")
print("  Applied in training OOF + testing"); mark_done("tta")

## 🎯 Step 24 — Threshold Optimization


In [ ]:
# STEP 24 (RESUMABLE)
if is_done("thresholds"):
    step_skip(24,"THRESHOLDS")
else:
    t0=step_start(24,"THRESHOLD OPTIMIZATION")
    op=np.load(str(ARTIFACT_DIR/"oof_preds.npy")); ol=np.load(str(ARTIFACT_DIR/"oof_labels.npy"))
    base_cls=reg_to_class(op); base_q=qwk(ol,base_cls)
    print(f"  Default thresholds {REG_THRESHOLDS}: QWK={base_q:.4f}")
    # Optimize
    def neg_qwk_thr(thr):
        c=np.digitize(np.clip(op,0,4),sorted(thr)); return -qwk(ol,c)
    res=minimize(neg_qwk_thr,REG_THRESHOLDS,method="Nelder-Mead",options={"maxiter":1000})
    opt_thr=sorted(res.x.tolist()); opt_cls=np.digitize(np.clip(op,0,4),opt_thr)
    opt_q=qwk(ol,opt_cls)
    print(f"  Optimised thresholds {[round(t,3) for t in opt_thr]}: QWK={opt_q:.4f}")
    # Force final (per Kaggle top-1%)
    final_thr=REG_THRESHOLDS; final_q=base_q
    if opt_q>base_q: final_thr=opt_thr; final_q=opt_q
    save_json({"default":REG_THRESHOLDS,"optimised":opt_thr,"final":final_thr,"qwk":float(final_q)},ARTIFACT_DIR/"thresholds.json")
    st_save("opt_qwk",float(final_q)); mark_done("thresholds")
    step_end(24,t0)

## 🧪 Step 25 — Final Testing


In [ ]:
# STEP 25 (RESUMABLE)
if is_done("test"):
    step_skip(25,"TESTING",f"QWK={st_get('test_qwk','?')}")
else:
    t0=step_start(25,"FINAL TESTING (ENSEMBLE + TTA)")
    df_test=df[df["fold"]==0].reset_index(drop=True)
    ens=np.zeros(len(df_test),dtype=np.float32); nm=0
    for fold in range(1,N_FOLDS):
        fc=CKPT_DIR/f"fold{fold}_best.pt"
        if not fc.exists(): continue
        ckpt=safe_load(fc,DEVICE); model=build_model(pretrained=False)
        model.load_state_dict(ckpt["model_state"])
        if "ema" in ckpt:
            for n,p in model.named_parameters():
                if n in ckpt["ema"]: p.data.copy_(ckpt["ema"][n])
        model.eval()
        fp=np.zeros(len(df_test),dtype=np.float32)
        with torch.no_grad():
            for ttf in build_tta_tf(384):
                ds=DRDataset(df_test,ttf,384); ld=make_loader(ds,BS_MAP[384],shuffle=False)
                bp=[]
                for im,_ in ld: bp.extend(model(im.to(DEVICE)).cpu().numpy().tolist())
                fp+=np.array(bp)
        fp/=len(build_tta_tf(384)); ens+=fp; nm+=1
        fc2=reg_to_class(fp); fq=qwk(df_test["diagnosis"].values,fc2)
        print(f"  Fold {fold}: QWK={fq:.4f}"); del model; gc.collect()
    if nm>0:
        ens/=nm; tl=df_test["diagnosis"].values; tp=reg_to_class(ens)
        tq=qwk(tl,tp); ta=acc_fn(tl,tp)
        print(f"\n  ✅ Ensemble QWK={tq:.4f} Acc={ta*100:.2f}%")
        st_save("test_qwk",float(tq)); st_save("test_acc",float(ta))
    mark_done("test"); step_end(25,t0)

## 📊 Step 26 — Metrics


In [ ]:
# STEP 26 (RESUMABLE)
if is_done("metrics"):
    step_skip(26,"METRICS")
else:
    t0=step_start(26,"METRICS")
    %matplotlib inline
    op=np.load(str(ARTIFACT_DIR/"oof_preds.npy")); ol=np.load(str(ARTIFACT_DIR/"oof_labels.npy"))
    pred=reg_to_class(op); oq=qwk(ol,pred); oa=acc_fn(ol,pred)
    pr=precision_score(ol,pred,average="weighted",zero_division=0)
    rc=recall_score(ol,pred,average="weighted",zero_division=0)
    f1v=f1_score(ol,pred,average="weighted",zero_division=0)
    m={"qwk":round(oq,4),"accuracy":round(oa,4),"precision":round(pr,4),"recall":round(rc,4),"f1":round(f1v,4)}
    for k,v in m.items(): print(f"  {k:12s}: {v}")
    fig,ax=plt.subplots(1,2,figsize=(16,6))
    cm=confusion_matrix(ol,pred)
    ConfusionMatrixDisplay(cm,display_labels=[f"G{i}" for i in range(5)]).plot(ax=ax[0],colorbar=False,cmap="Blues")
    ax[0].set_title(f"QWK={oq:.4f}",fontweight="bold")
    pcr=cm.diagonal()/np.maximum(cm.sum(axis=1),1)
    ax[1].bar([GRADE_MAP[i] for i in range(5)],pcr,color=GRADE_COLORS)
    ax[1].axhline(0.85,color="red",ls="--"); ax[1].set_ylim(0,1.05)
    plt.tight_layout(); plt.savefig(PLOT_DIR/"metrics.png",dpi=150); plt.show()
    print(f"\n{classification_report(ol,pred,target_names=[GRADE_MAP[i] for i in range(5)])}")
    save_json(m,LOG_DIR/"metrics.json"); mark_done("metrics"); step_end(26,t0)

## 🔥 Step 27 — Grad-CAM++


In [ ]:
# STEP 27 (RESUMABLE)
if is_done("explainability"):
    step_skip(27,"GRAD-CAM++")
else:
    t0=step_start(27,"GRAD-CAM++")
    %matplotlib inline
    try:
        from pytorch_grad_cam import GradCAMPlusPlus
        from pytorch_grad_cam.utils.image import show_cam_on_image
        from pytorch_grad_cam.utils.model_targets import RawScoresOutputTarget
        fq=load_json(ARTIFACT_DIR/"fold_qwks.json"); bf=int(np.argmax(fq))
        ckpt=safe_load(CKPT_DIR/f"fold{bf}_best.pt","cpu")
        gm=build_model(pretrained=False); gm.load_state_dict(ckpt["model_state"]); gm.eval().to("cpu")
        cam=GradCAMPlusPlus(model=gm,target_layers=[gm.backbone.blocks[-1][-1]])
        fig,axes=plt.subplots(5,4,figsize=(14,15))
        for g in range(5):
            samp=df[df["diagnosis"]==g].sample(min(2,(df["diagnosis"]==g).sum()),random_state=SEED)
            for j,(_, row) in enumerate(samp.iterrows()):
                img=preprocess_fundus(row["image_path"],384)
                inp=build_val_tf(384)(image=img)["image"].unsqueeze(0)
                gs=cam(input_tensor=inp,targets=[RawScoresOutputTarget()])
                ci=show_cam_on_image(img.astype(np.float32)/255.,gs[0],use_rgb=True)
                axes[g][j*2].imshow(img); axes[g][j*2].axis("off")
                axes[g][j*2+1].imshow(ci); axes[g][j*2+1].axis("off")
        plt.suptitle(f"Grad-CAM++ Fold {bf}",fontweight="bold")
        plt.tight_layout(); plt.savefig(PLOT_DIR/"gradcam.png",dpi=130); plt.show()
        del gm; gc.collect(); mark_done("explainability"); step_end(27,t0)
    except ImportError: print("  pip install grad-cam"); print("="*68)

## 📦 Step 28 — Model Export


In [ ]:
# STEP 28 (RESUMABLE)
import shutil as _sh
if is_done("export"):
    step_skip(28,"EXPORT")
else:
    t0=step_start(28,"MODEL EXPORT")
    EXPORT_DIR.mkdir(exist_ok=True)
    fq=load_json(ARTIFACT_DIR/"fold_qwks.json"); bf=int(np.argmax(fq))
    src=CKPT_DIR/f"fold{bf}_best.pt"
    if src.exists(): _sh.copy2(src,EXPORT_DIR/"best_model.pt"); print(f"  best_model.pt fold {bf}")
    save_json({"thresholds":REG_THRESHOLDS,"backbone":BACKBONE,"approach":"regression"},EXPORT_DIR/"config.json")
    mark_done("export"); step_end(28,t0)

## 🌐 Step 29 — Deployment (Streamlit + Validator)


In [ ]:
# STEP 29 — Deployment (RESUMABLE)
import shutil as _sh
if is_done("deployment"):
    step_skip(29, "DEPLOYMENT")
else:
    t0 = step_start(29, "DEPLOYMENT")
    DEPLOY_DIR.mkdir(exist_ok=True)
    for f in EXPORT_DIR.glob("*"):
        if f.is_file(): _sh.copy2(f, DEPLOY_DIR / f.name)

    # model_utils.py
    mu_code = "import numpy as np, cv2, torch, torch.nn as nn, torch.nn.functional as F, timm
"
    mu_code += "import albumentations as A
from albumentations.pytorch import ToTensorV2
"
    mu_code += "BACKBONE='tf_efficientnetv2_b1'; IMG_SIZE=384; THRESHOLDS=[0.7,1.5,2.5,3.5]
"
    (DEPLOY_DIR / "model_utils.py").write_text(mu_code)
    print("  model_utils.py")

    # validator.py
    val_code = "import cv2, numpy as np
"
    val_code += "def is_retinal(img):
"
    val_code += "    if img.shape[0]<100 or img.shape[1]<100: return False
"
    val_code += "    gray=cv2.cvtColor(img,cv2.COLOR_RGB2GRAY)
"
    val_code += "    if float(gray.mean())<5: return False
"
    val_code += "    return True  # fail-safe
"
    (DEPLOY_DIR / "validator.py").write_text(val_code)
    print("  validator.py")

    req = "torch>=2.1
torchvision
timm>=1.0.0
albumentations>=1.4.0
opencv-python-headless
streamlit
numpy
"
    (DEPLOY_DIR / "requirements.txt").write_text(req)
    print("  requirements.txt")

    mark_done("deployment")
    step_end(29, t0)


## 🧠 Step 30 — Inference + Final Summary


In [ ]:
# STEP 30 — Final Summary
if not is_done("inference"): mark_done("inference")
state=st_load()
print("="*68)
print("  DIABETIC RETINOPATHY — v21 FINAL SUMMARY")
print("="*68)
print(f"  Approach  : REGRESSION (SmoothL1Loss)")
print(f"  Backbone  : {BACKBONE}")
print(f"  Device    : {DEVICE.upper()}")
print(f"  Dataset   : {len(df):,}")
print(f"  Thresholds: {REG_THRESHOLDS}")

if (ARTIFACT_DIR/"fold_qwks.json").exists():
    fq=load_json(ARTIFACT_DIR/"fold_qwks.json")
    for i,q in enumerate(fq): print(f"    Fold {i}: QWK={q:.4f}")
    print(f"    Mean  : {np.mean(fq):.4f} ± {np.std(fq):.4f}")

print(f"\n  OOF QWK  : {state.get('oof_qwk','N/A')}")
print(f"  Opt QWK  : {state.get('opt_qwk','N/A')}")
print(f"  Test QWK : {state.get('test_qwk','N/A')}")
ta=state.get('test_acc')
print(f"  Test Acc : {float(ta)*100:.2f}%" if ta else "  Test Acc : N/A")
if (LOG_DIR/"metrics.json").exists():
    m=load_json(LOG_DIR/"metrics.json")
    print(f"  F1       : {m.get('f1','N/A')}")
    print(f"  Precision: {m.get('precision','N/A')}")
    print(f"  Recall   : {m.get('recall','N/A')}")

done=sorted(FLAG_DIR.glob("*.done"))
print(f"\n  ✅ {len(done)} steps completed")
print(f"  Export: {EXPORT_DIR}")
print(f"  Deploy: {DEPLOY_DIR}")
print("="*68)
print("  ⚠️ RESEARCH USE ONLY")
print("="*68)